# 🤰 Onamiz — AI Model v4 (42 xavf, O'zbekiston + Global)

## Aniqlanadigan xavflar (42 ta):
### 1-Trimest (11 ta)
- Ektopik homila, Spontan abort, Hyperemesis gravidarum
- UTI/Pielonefrit, Infeksiya/Sepsis, Anemiya
- Takroriy abort, **Tireoid kasalligi (yangi)**, **Rh mos kelmaslik (yangi)**
- O'zbekiston: erta homiladorlik, STI xavfi

### 2-Trimest (11 ta)
- Preeklampsia, Og'ir preeklampsia, HELLP sindrom
- Fetal distress, PPROM, Gestatsion diabet
- Suyuqlik ushlanishi, **Plasenta qoplag'i (yangi)**
- **Bachadon bo'yni zaiflik (yangi)**, **Ko'p suv (yangi)**

### 3-Trimest (13 ta)
- Eklampsiya, Homila gipoksiyasi, Muddatidan oldin tug'ruq
- Plasenta ajralishi, O'pka shishi, IUGR
- **Jigar xolestazi ICP (yangi)** — safroo kislotasi >40µmol/L
- **Ko'p suv (yangi)**, **42+ hafta post-term (yangi)**
- **GDM nazorat (yangi)**, **Plasenta qoplag'i T3 (yangi)**

### Tug'ruqdan keyin (3 ta) + O'zbekiston xususiy (4 ta)
- EPDS depressiya, Obstetrik qon ketish, Anemiya

**Klinik manbalar:** WHO ANC 2016 | ACOG 2023 | FIGO 2022 | SMFM #53 | WHO/IADPSG 2013 | ATA 2017 | Andijan Study 2025

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm joblib matplotlib seaborn imbalanced-learn shap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/onamiz_data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/onamiz_models', exist_ok=True)
print('✅ Drive ulandi')

In [ ]:
import warnings, os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Tayyor')

## 🌍 BLOK 1 — Global dataset (WHO/ACOG/FIGO/SMFM/ATA)

In [ ]:
np.random.seed(RANDOM_STATE)

# ============================================================
# BARCHA FEATURE NOMLARI (42 xavfni qamrab oladi)
# ============================================================
FEATURE_COLS = [
    # Asosiy
    'trimester_enc', 'age', 'gestational_week',
    # 1-TRIMEST — WHO/ACOG
    'vaginal_bleeding',       # 0=yo'q, 1=ozgina, 2=ko'p
    'one_sided_pain',         # 0=yo'q, 1=ha (ektopik)
    'nausea_severity',        # 0-4
    'urinary_burning',        # 0=yo'q, 1=ha (UTI/STI)
    'fever',                  # 0=normal, 1=subfebril, 2=yuqori
    'dizziness',              # 0-2 (anemiya)
    'prev_miscarriage',       # 0=yo'q, 1=1x, 2=2+
    'thyroid_symptoms',       # 0=yo'q, 1=charchoq, 2=yurak tezlashdi, 3=ikkalasi — ATA 2017
    'rh_checked',             # 0=tekshirilmagan, 1=tekshirilgan — ACOG PB-181
    # 2-TRIMEST — FIGO/ACOG
    'headache_severity',      # 0-2
    'visual_disturbance',     # 0=yo'q, 1=ha (eklampsiya)
    'edema_level',            # 0-3
    'fetal_movement',         # 0=yaxshi, 1=kam, 2=yo'q
    'epigastric_pain',        # 0-2 (HELLP)
    'sudden_weight_gain',     # 0=yo'q, 1=ha (3+kg/hafta)
    'fluid_leaking',          # 0=yo'q, 1=ha (PPROM)
    'painless_bleeding',      # 0=yo'q, 1=dog', 2=qon — Plasenta previa
    'fasting_glucose',        # 0=normal, 1=5.1-6.9, 2=7+, 3=bilmaydi — WHO/IADPSG
    'cervix_short',           # 0=yo'q, 1=ha (<25mm) — ACOG PB-142
    'belly_very_large',       # 0=yo'q, 1=ha (polihidramnios)
    # 3-TRIMEST — WHO/ACOG/SMFM
    'fetal_movement_t3',      # 0=yaxshi, 1=kam, 2=yo'q
    'contractions',           # 0=yo'q, 1=ba'zan, 2=muntazam
    'bleeding_with_pain',     # 0=yo'q, 1=ha (plasenta ajralishi)
    'shortness_of_breath',    # 0-2
    'itching_palms_soles',    # 0=yo'q, 1=ozgina, 2=kuchli, 3=uxlay olmaydi — ICP SMFM#53
    'post_term',              # 0=yo'q, 1=41 hafta, 2=42+ hafta — ACOG PB-146
    # Vital
    'systolic_bp',
    'diastolic_bp',
    'heart_rate',
    # O'ZBEKISTON XUSUSIY — SSV/Demografiya.uz/Andijan 2025
    'anemia_level',           # 0-3 (WHO Uzbekistan: 30-74%)
    'iron_supplement',        # 0=yo'q, 1=ha
    'parity',                 # 0=birinchi, 1=1-2x, 2=3+
    'prenatal_visits',        # 0=hech, 1=1-3, 2=4+
    'nutrition_poor',         # 0=yaxshi, 1=yomon
    'rural',                  # 0=shahar, 1=qishloq
    'pph_history',            # 0=yo'q, 1=ha (Andijan: 24% qon ketish)
]

print(f'Jami features: {len(FEATURE_COLS)} ta')
print('Yangi features (v4):', ['thyroid_symptoms','rh_checked','painless_bleeding',
                                'fasting_glucose','cervix_short','belly_very_large',
                                'itching_palms_soles','post_term'])

In [ ]:
def generate_global_dataset(n=3000):
    """WHO/ACOG/FIGO/SMFM/ATA protokollariga asoslangan global dataset."""
    records = []
    for _ in range(n):
        trimester = np.random.choice(['T1','T2','T3'], p=[0.35,0.35,0.30])
        age = np.random.randint(16, 45)
        week = {'T1':np.random.randint(4,13),'T2':np.random.randint(13,27),'T3':np.random.randint(27,41)}[trimester]

        # --- 1-TRIMEST ---
        vaginal_bleeding   = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        one_sided_pain     = np.random.choice([0,1],   p=[0.93,0.07])
        nausea_severity    = np.random.choice([0,1,2,3,4], p=[0.20,0.30,0.30,0.15,0.05])
        urinary_burning    = np.random.choice([0,1],   p=[0.85,0.15])
        fever              = np.random.choice([0,1,2], p=[0.85,0.10,0.05])
        dizziness          = np.random.choice([0,1,2], p=[0.70,0.20,0.10])
        prev_miscarriage   = np.random.choice([0,1,2], p=[0.75,0.15,0.10])
        # YANGI: Tireoid — ATA 2017 (5-10% homiladorlarda)
        thyroid_symptoms   = np.random.choice([0,1,2,3], p=[0.88,0.06,0.04,0.02])
        # YANGI: Rh tekshiruvi
        rh_checked         = np.random.choice([0,1], p=[0.15,0.85])

        # --- 2-TRIMEST ---
        headache_severity  = np.random.choice([0,1,2], p=[0.70,0.20,0.10])
        visual_disturbance = np.random.choice([0,1],   p=[0.95,0.05])
        edema_level        = np.random.choice([0,1,2,3], p=[0.55,0.25,0.15,0.05])
        fetal_movement     = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        epigastric_pain    = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        sudden_weight_gain = np.random.choice([0,1],   p=[0.90,0.10])
        fluid_leaking      = np.random.choice([0,1],   p=[0.97,0.03])
        # YANGI: Plasenta previa — og'riqsiz qon (0.5% homilada)
        painless_bleeding  = np.random.choice([0,1,2], p=[0.93,0.05,0.02])
        # YANGI: Gestatsion diabet — WHO/IADPSG 2013 (7-10%)
        fasting_glucose    = np.random.choice([0,1,2,3], p=[0.72,0.12,0.04,0.12])
        # YANGI: Bachadon bo'yni zaiflik (1-2%)
        cervix_short       = np.random.choice([0,1],   p=[0.98,0.02])
        # YANGI: Polihidramnios (1-2%)
        belly_very_large   = np.random.choice([0,1],   p=[0.97,0.03])

        # --- 3-TRIMEST ---
        fetal_movement_t3  = np.random.choice([0,1,2], p=[0.78,0.17,0.05])
        contractions       = np.random.choice([0,1,2], p=[0.75,0.17,0.08])
        bleeding_with_pain = np.random.choice([0,1],   p=[0.97,0.03])
        shortness_of_breath= np.random.choice([0,1,2], p=[0.75,0.20,0.05])
        # YANGI: ICP — Jigar xolestazi — SMFM #53 (1-2%)
        itching_palms_soles= np.random.choice([0,1,2,3], p=[0.88,0.06,0.04,0.02])
        # YANGI: Post-term (10% 41+ hafta, 3% 42+ hafta)
        post_term          = np.random.choice([0,1,2], p=[0.87,0.10,0.03])

        # Vital
        systolic_bp  = np.random.normal(115, 15)
        diastolic_bp = np.random.normal(75, 10)
        heart_rate   = np.random.normal(82, 12)

        # O'zbekiston
        anemia_level    = np.random.choice([0,1,2,3], p=[0.60,0.25,0.12,0.03])
        iron_supplement = np.random.choice([0,1], p=[0.50,0.50])
        parity          = np.random.choice([0,1,2], p=[0.40,0.40,0.20])
        prenatal_visits = np.random.choice([0,1,2], p=[0.10,0.40,0.50])
        nutrition_poor  = np.random.choice([0,1], p=[0.75,0.25])
        rural           = np.random.choice([0,1], p=[0.60,0.40])
        pph_history     = np.random.choice([0,1], p=[0.92,0.08])

        # ============================================================
        # XAVF HISOBLASH — 42 xavf uchun
        # ============================================================
        s = 0

        # FAVQULODDA (darhol)
        if vaginal_bleeding == 2:      s += 10  # ko'p qon
        if one_sided_pain == 1:        s += 10  # ektopik
        if visual_disturbance == 1:    s += 10  # eklampsiya
        if fetal_movement == 2:        s += 10  # homila harakatsiz
        if fluid_leaking == 1:         s += 10  # PPROM
        if bleeding_with_pain == 1:    s += 10  # plasenta ajralishi
        if fetal_movement_t3 == 2:     s += 10  # gipoksiya T3
        if painless_bleeding == 2:     s += 10  # plasenta previa (qon)

        # YUQORI XAVF (bugun boring)
        if headache_severity == 2:     s += 5
        if epigastric_pain == 2:       s += 5   # HELLP
        if contractions == 2:          s += 5   # erta tug'ruq
        if systolic_bp >= 160:         s += 5   # og'ir preeklampsia
        if systolic_bp >= 140:         s += 4
        if diastolic_bp >= 90:         s += 4
        if fever == 2:                 s += 4
        if nausea_severity == 4:       s += 4   # hyperemesis
        if shortness_of_breath == 2:   s += 4   # o'pka shishi
        if anemia_level == 3:          s += 4   # og'ir anemiya
        if itching_palms_soles == 3:   s += 4   # ICP og'ir (SMFM)
        if post_term == 2:             s += 4   # 42+ hafta
        if fasting_glucose == 2:       s += 4   # GDM og'ir
        if cervix_short == 1:          s += 4   # bachadon bo'yni
        if pph_history == 1 and parity >= 1: s += 3  # qon ketish tarixi

        # O'RTA XAVF (bu hafta boring)
        if vaginal_bleeding == 1:      s += 3
        if nausea_severity == 3:       s += 3
        if urinary_burning == 1:       s += 3   # UTI
        if dizziness == 2:             s += 3
        if fetal_movement == 1:        s += 3
        if fetal_movement_t3 == 1:     s += 3
        if edema_level == 3:           s += 3
        if anemia_level == 2:          s += 2
        if headache_severity == 1:     s += 2
        if edema_level == 2:           s += 2
        if sudden_weight_gain == 1:    s += 2
        if prev_miscarriage == 2:      s += 2
        if fever == 1:                 s += 2
        if contractions == 1:          s += 2
        if epigastric_pain == 1:       s += 2
        if shortness_of_breath == 1:   s += 2
        if pph_history == 1:           s += 2
        if thyroid_symptoms >= 2:      s += 2   # tireoid (ATA)
        if painless_bleeding == 1:     s += 3   # plasenta previa dog'
        if fasting_glucose == 1:       s += 2   # GDM chegaraviy
        if fasting_glucose == 3:       s += 1   # bilmaydi — xavfli
        if belly_very_large == 1:      s += 2   # polihidramnios
        if itching_palms_soles == 2:   s += 2   # ICP o'rta
        if post_term == 1:             s += 2   # 41 hafta

        # PAST OMILLAR
        if age < 18 or age > 40:       s += 1
        if prev_miscarriage == 1:      s += 1
        if anemia_level == 1:          s += 1
        if prenatal_visits == 0:       s += 1
        if nutrition_poor == 1:        s += 1
        if rh_checked == 0:            s += 1   # Rh tekshirilmagan
        if thyroid_symptoms == 1:      s += 1
        if itching_palms_soles == 1:   s += 1
        if iron_supplement == 0 and anemia_level >= 1: s += 1

        risk = 'emergency' if s>=10 else 'high' if s>=6 else 'medium' if s>=3 else 'low'

        records.append({
            'trimester':trimester,'age':age,'gestational_week':week,
            'vaginal_bleeding':vaginal_bleeding,'one_sided_pain':one_sided_pain,
            'nausea_severity':nausea_severity,'urinary_burning':urinary_burning,
            'fever':fever,'dizziness':dizziness,'prev_miscarriage':prev_miscarriage,
            'thyroid_symptoms':thyroid_symptoms,'rh_checked':rh_checked,
            'headache_severity':headache_severity,'visual_disturbance':visual_disturbance,
            'edema_level':edema_level,'fetal_movement':fetal_movement,
            'epigastric_pain':epigastric_pain,'sudden_weight_gain':sudden_weight_gain,
            'fluid_leaking':fluid_leaking,'painless_bleeding':painless_bleeding,
            'fasting_glucose':fasting_glucose,'cervix_short':cervix_short,
            'belly_very_large':belly_very_large,
            'fetal_movement_t3':fetal_movement_t3,'contractions':contractions,
            'bleeding_with_pain':bleeding_with_pain,'shortness_of_breath':shortness_of_breath,
            'itching_palms_soles':itching_palms_soles,'post_term':post_term,
            'systolic_bp':round(systolic_bp,1),'diastolic_bp':round(diastolic_bp,1),
            'heart_rate':round(heart_rate,1),
            'anemia_level':anemia_level,'iron_supplement':iron_supplement,
            'parity':parity,'prenatal_visits':prenatal_visits,
            'nutrition_poor':nutrition_poor,'rural':rural,'pph_history':pph_history,
            'source':'global','risk_level':risk
        })
    return pd.DataFrame(records)

df_global = generate_global_dataset(3000)
print('Global dataset:', df_global.shape)
print(df_global['risk_level'].value_counts())

## 🇺🇿 BLOK 2 — O'zbekiston dataseti

In [ ]:

def generate_uzbekistan_dataset(n=2000):
    """
    O'zbekiston-xususiy dataset.
    Andijan 2025 + WHO Uzbekistan + Demografiya.uz asosida.
    """
    records = []
    for _ in range(n):
        trimester = np.random.choice(['T1','T2','T3'], p=[0.35,0.35,0.30])

        # Yosh — O'zbekiston: yosh onalar ko'proq (normal taqsimot, markaz=22)
        age = int(np.clip(np.random.normal(22, 7), 15, 44))
        week = {'T1':np.random.randint(4,13),'T2':np.random.randint(13,27),'T3':np.random.randint(27,41)}[trimester]

        # ANEMIYA — O'zbekiston: o'smirlarda 74%, kattalarda 60%
        anemia_level = np.random.choice([0,1,2,3],
            p=[0.26,0.35,0.28,0.11] if age<19 else [0.40,0.35,0.20,0.05])

        iron_supplement = np.random.choice([0,1], p=[0.65,0.35])
        rural = np.random.choice([0,1], p=[0.35,0.65])
        parity = np.random.choice([0,1,2], p=[0.25,0.35,0.40] if rural else [0.45,0.40,0.15])
        prenatal_visits = np.random.choice([0,1,2], p=[0.25,0.45,0.30] if rural else [0.05,0.30,0.65])
        nutrition_poor = np.random.choice([0,1], p=[0.55,0.45])
        pph_history = np.random.choice([0,1], p=[0.82,0.18] if parity>=2 else [0.93,0.07])
        dizziness = np.random.choice([0,1,2], p=[0.30,0.40,0.30] if anemia_level>=2 else [0.70,0.20,0.10])
        urinary_burning = np.random.choice([0,1], p=[0.78,0.22] if age<19 else [0.85,0.15])
        contractions = np.random.choice([0,1,2], p=[0.70,0.20,0.10] if age<19 else [0.76,0.17,0.07])
        fasting_glucose = np.random.choice([0,1,2,3], p=[0.60,0.18,0.08,0.14])
        thyroid_symptoms = np.random.choice([0,1,2,3], p=[0.86,0.07,0.05,0.02])
        rh_checked = np.random.choice([0,1], p=[0.30,0.70])

        vaginal_bleeding    = np.random.choice([0,1,2], p=[0.78,0.16,0.06])
        one_sided_pain      = np.random.choice([0,1],   p=[0.93,0.07])
        nausea_severity     = np.random.choice([0,1,2,3,4], p=[0.18,0.28,0.32,0.16,0.06])
        fever               = np.random.choice([0,1,2], p=[0.83,0.12,0.05])
        prev_miscarriage    = np.random.choice([0,1,2], p=[0.70,0.18,0.12])
        headache_severity   = np.random.choice([0,1,2], p=[0.68,0.22,0.10])
        visual_disturbance  = np.random.choice([0,1],   p=[0.95,0.05])
        edema_level         = np.random.choice([0,1,2,3], p=[0.52,0.28,0.15,0.05])
        fetal_movement      = np.random.choice([0,1,2], p=[0.80,0.15,0.05])
        epigastric_pain     = np.random.choice([0,1,2], p=[0.80,0.14,0.06])
        sudden_weight_gain  = np.random.choice([0,1],   p=[0.89,0.11])
        fluid_leaking       = np.random.choice([0,1],   p=[0.97,0.03])
        painless_bleeding   = np.random.choice([0,1,2], p=[0.92,0.06,0.02])
        cervix_short        = np.random.choice([0,1],   p=[0.98,0.02])
        belly_very_large    = np.random.choice([0,1],   p=[0.96,0.04])
        fetal_movement_t3   = np.random.choice([0,1,2], p=[0.76,0.18,0.06])
        bleeding_with_pain  = np.random.choice([0,1],   p=[0.96,0.04])
        shortness_of_breath = np.random.choice([0,1,2], p=[0.75,0.20,0.05])
        itching_palms_soles = np.random.choice([0,1,2,3], p=[0.88,0.06,0.04,0.02])
        post_term           = np.random.choice([0,1,2], p=[0.87,0.10,0.03])
        systolic_bp  = round(np.random.normal(118, 16), 1)
        diastolic_bp = round(np.random.normal(76, 11), 1)
        heart_rate   = round(np.random.normal(84 + anemia_level*3, 13), 1)

        # XAVF HISOBLASH
        s = 0
        if vaginal_bleeding==2:      s+=10
        if one_sided_pain==1:        s+=10
        if visual_disturbance==1:    s+=10
        if fetal_movement==2:        s+=10
        if fluid_leaking==1:         s+=10
        if bleeding_with_pain==1:    s+=10
        if fetal_movement_t3==2:     s+=10
        if painless_bleeding==2:     s+=10
        if headache_severity==2:     s+=5
        if epigastric_pain==2:       s+=5
        if contractions==2:          s+=5
        if systolic_bp>=160:         s+=5
        if systolic_bp>=140:         s+=4
        if diastolic_bp>=90:         s+=4
        if fever==2:                 s+=4
        if anemia_level==3:          s+=4
        if itching_palms_soles==3:   s+=4
        if post_term==2:             s+=4
        if fasting_glucose==2:       s+=4
        if cervix_short==1:          s+=4
        if nausea_severity==4:       s+=4
        if shortness_of_breath==2:   s+=4
        if pph_history==1 and parity>=1: s+=3
        if vaginal_bleeding==1:      s+=3
        if nausea_severity==3:       s+=3
        if urinary_burning==1:       s+=3
        if dizziness==2:             s+=3
        if fetal_movement==1:        s+=3
        if fetal_movement_t3==1:     s+=3
        if edema_level==3:           s+=3
        if painless_bleeding==1:     s+=3
        if anemia_level==2:          s+=2
        if headache_severity==1:     s+=2
        if edema_level==2:           s+=2
        if sudden_weight_gain==1:    s+=2
        if prev_miscarriage==2:      s+=2
        if fever==1:                 s+=2
        if contractions==1:          s+=2
        if epigastric_pain==1:       s+=2
        if shortness_of_breath==1:   s+=2
        if pph_history==1:           s+=2
        if thyroid_symptoms>=2:      s+=2
        if fasting_glucose==1:       s+=2
        if belly_very_large==1:      s+=2
        if itching_palms_soles==2:   s+=2
        if post_term==1:             s+=2
        if age<18:                   s+=2
        if age>40:                   s+=1
        if parity>=2:                s+=1
        if rural==1 and prenatal_visits==0: s+=2
        if iron_supplement==0 and anemia_level>=1: s+=1
        if nutrition_poor==1:        s+=1
        if prev_miscarriage==1:      s+=1
        if anemia_level==1:          s+=1
        if prenatal_visits==0:       s+=1
        if rh_checked==0:            s+=1
        if thyroid_symptoms==1:      s+=1
        if itching_palms_soles==1:   s+=1
        if fasting_glucose==3:       s+=1

        risk = 'emergency' if s>=10 else 'high' if s>=6 else 'medium' if s>=3 else 'low'

        records.append({
            'trimester':trimester,'age':age,'gestational_week':week,
            'vaginal_bleeding':vaginal_bleeding,'one_sided_pain':one_sided_pain,
            'nausea_severity':nausea_severity,'urinary_burning':urinary_burning,
            'fever':fever,'dizziness':dizziness,'prev_miscarriage':prev_miscarriage,
            'thyroid_symptoms':thyroid_symptoms,'rh_checked':rh_checked,
            'headache_severity':headache_severity,'visual_disturbance':visual_disturbance,
            'edema_level':edema_level,'fetal_movement':fetal_movement,
            'epigastric_pain':epigastric_pain,'sudden_weight_gain':sudden_weight_gain,
            'fluid_leaking':fluid_leaking,'painless_bleeding':painless_bleeding,
            'fasting_glucose':fasting_glucose,'cervix_short':cervix_short,
            'belly_very_large':belly_very_large,
            'fetal_movement_t3':fetal_movement_t3,'contractions':contractions,
            'bleeding_with_pain':bleeding_with_pain,'shortness_of_breath':shortness_of_breath,
            'itching_palms_soles':itching_palms_soles,'post_term':post_term,
            'systolic_bp':systolic_bp,'diastolic_bp':diastolic_bp,'heart_rate':heart_rate,
            'anemia_level':anemia_level,'iron_supplement':iron_supplement,
            'parity':parity,'prenatal_visits':prenatal_visits,
            'nutrition_poor':nutrition_poor,'rural':rural,'pph_history':pph_history,
            'source':'uzbekistan','risk_level':risk
        })
    return pd.DataFrame(records)

df_uzbek = generate_uzbekistan_dataset(2000)
print("O'zbekiston dataset:", df_uzbek.shape)
print(df_uzbek['risk_level'].value_counts())
print(f"\nAnemiya: {(df_uzbek.anemia_level>0).mean():.1%}")
print(f"Yosh onalar (<19): {(df_uzbek.age<19).mean():.1%}")
print(f"GDM xavfi: {(df_uzbek.fasting_glucose>0).mean():.1%}")
print(f"ICP xavfi: {(df_uzbek.itching_palms_soles>0).mean():.1%}")


## 🔗 BLOK 3 — Birlashtirish va tahlil

In [ ]:
# Kaggle dataseti (mavjud bo'lsa)
KAGGLE_PATH = '/content/drive/MyDrive/onamiz_data/Maternal Health Risk Data Set.csv'

df_all = pd.concat([df_global, df_uzbek], ignore_index=True)

if os.path.exists(KAGGLE_PATH):
    kdf = pd.read_csv(KAGGLE_PATH)
    kdf_mapped = pd.DataFrame({
        'trimester':'T2','age':kdf['Age'],'gestational_week':20,
        'systolic_bp':kdf['SystolicBP'],'diastolic_bp':kdf['DiastolicBP'],
        'heart_rate':kdf['HeartRate'],'fever':(kdf['BodyTemp']>99.5).astype(int),
        **{c:0 for c in FEATURE_COLS if c not in
           ['trimester_enc','age','gestational_week','systolic_bp','diastolic_bp','heart_rate','fever']},
        'source':'kaggle',
        'risk_level':kdf['RiskLevel'].map({'low risk':'low','mid risk':'medium','high risk':'high'})
    })
    df_all = pd.concat([df_all, kdf_mapped], ignore_index=True)
    print(f'Kaggle qoshildi: {len(kdf)} ta')

df_all = df_all.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'\nJami dataset: {df_all.shape}')
print(f'Manbalar: {df_all.source.value_counts().to_dict()}')
print(f'\nXavf taqsimoti:\n{df_all.risk_level.value_counts()}')

# Vizual tahlil
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
colors = {'low':'#27ae60','medium':'#f39c12','high':'#e74c3c','emergency':'#8e44ad'}
order  = ['low','medium','high','emergency']

# 1. Umumiy taqsimot
df_all.risk_level.value_counts()[order].plot(kind='bar', ax=axes[0,0],
    color=[colors[c] for c in order], edgecolor='white')
axes[0,0].set_title('Umumiy xavf taqsimoti', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=0)

# 2. Global vs O'zbekiston
comp = df_all.groupby(['source','risk_level']).size().unstack(fill_value=0)[order]
comp.div(comp.sum(axis=1),axis=0).plot(kind='bar', ax=axes[0,1],
    color=[colors[c] for c in order], edgecolor='white')
axes[0,1].set_title("Global vs O'zbekiston", fontweight='bold')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))
axes[0,1].tick_params(axis='x', rotation=0)

# 3. Yosh guruhi
age_b = pd.cut(df_uzbek['age'], bins=[14,18,25,35,50], labels=['15-18','19-25','26-35','36+'])
df_uzbek.groupby([age_b,'risk_level']).size().unstack(fill_value=0)[order]\
    .div(df_uzbek.groupby(age_b).size(),axis=0)\
    .plot(kind='bar', ax=axes[0,2], color=[colors[c] for c in order], edgecolor='white')
axes[0,2].set_title("Yosh guruhi (O'zbekiston)", fontweight='bold')
axes[0,2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))
axes[0,2].tick_params(axis='x', rotation=0)

# 4. Anemiya
df_uzbek.groupby(['anemia_level','risk_level']).size().unstack(fill_value=0)[order]\
    .div(df_uzbek.groupby('anemia_level').size(),axis=0)\
    .plot(kind='bar', ax=axes[1,0], color=[colors[c] for c in order], edgecolor='white')
axes[1,0].set_title('Anemiya darajasi (O\'zbekiston)', fontweight='bold')
axes[1,0].set_xticklabels(["Yo'q",'Yengil',"O'rta","Og'ir"], rotation=0)
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))

# 5. GDM
df_all.groupby(['fasting_glucose','risk_level']).size().unstack(fill_value=0)[order]\
    .div(df_all.groupby('fasting_glucose').size(),axis=0)\
    .plot(kind='bar', ax=axes[1,1], color=[colors[c] for c in order], edgecolor='white')
axes[1,1].set_title('Qon shakari va xavf', fontweight='bold')
axes[1,1].set_xticklabels(['Normal','Chegaraviy','Yuqori','Bilmaydi'], rotation=0)
axes[1,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))

# 6. ICP
df_all.groupby(['itching_palms_soles','risk_level']).size().unstack(fill_value=0)[order]\
    .div(df_all.groupby('itching_palms_soles').size(),axis=0)\
    .plot(kind='bar', ax=axes[1,2], color=[colors[c] for c in order], edgecolor='white')
axes[1,2].set_title('ICP (Qichishish) va xavf', fontweight='bold')
axes[1,2].set_xticklabels(["Yo'q",'Ozgina','Kuchli','Uxlay olmaydi'], rotation=0)
axes[1,2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))

plt.suptitle('Onamiz v4 — Dataset tahlili', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/v4_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏋️ BLOK 4 — Model o'qitish

In [ ]:
le_trim = LabelEncoder()
df_all['trimester_enc'] = le_trim.fit_transform(df_all['trimester'])
le_y = LabelEncoder()
df_all['risk_enc'] = le_y.fit_transform(df_all['risk_level'])
print('Sinflar:', dict(enumerate(le_y.classes_)))
print(f'Features: {len(FEATURE_COLS)} ta')

X = df_all[FEATURE_COLS].fillna(0)
y = df_all['risk_enc']

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

smote = SMOTE(random_state=RANDOM_STATE)
X_bal, y_bal = smote.fit_resample(X_sc, y)
print('SMOTE keyin:', pd.Series(le_y.inverse_transform(y_bal)).value_counts().to_dict())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=RANDOM_STATE)
print(f'Train: {X_tr.shape[0]} | Test: {X_te.shape[0]}')

# O'qitish
candidates = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':      XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM':     LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}
print('\n' + '='*55)
for name, model in candidates.items():
    sc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = {'mean': sc.mean(), 'std': sc.std()}
    print(f'  {name:15s}  F1={sc.mean():.4f} (±{sc.std():.4f})')

best_name = max(results, key=lambda k: results[k]['mean'])
print(f'\n  🏆 {best_name} tanlandi')

best_model = candidates[best_name]
best_model.fit(X_tr, y_tr)
y_pred = best_model.predict(X_te)
acc = accuracy_score(y_te, y_pred)
f1  = f1_score(y_te, y_pred, average='macro')
print(f'\nTest Accuracy: {acc:.4f}')
print(f'Test F1-macro: {f1:.4f}')
print(classification_report(y_te, y_pred, target_names=le_y.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=axes[0])
axes[0].set_title(f'Onamiz v4 — {best_name}\nAcc={acc:.3f} | F1={f1:.3f}', fontweight='bold')
axes[0].set_xlabel('Bashorat'); axes[0].set_ylabel('Haqiqiy')

# Feature importance — top 20
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).nlargest(20)
    colors_imp = ['#e74c3c' if any(x in f for x in
        ['thyroid','rh_','painless','fasting','cervix','itching','post_term','belly_large'])
        else '#3498db' for f in imp.index]
    imp.plot(kind='barh', ax=axes[1], color=colors_imp)
    axes[1].set_title('Top 20 muhim feature\n(qizil = yangi qoshilganlar)', fontweight='bold')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/content/v4_model_results.png', dpi=150)
plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(best_model)
sv = explainer.shap_values(X_te[:300])
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv[0] if isinstance(sv, list) else sv,
    X_te[:300], feature_names=FEATURE_COLS, show=False, plot_size=(10,8)
)
plt.title('SHAP — Qaysi simptom koproq tasir qiladi?', fontsize=13)
plt.tight_layout()
plt.savefig('/content/v4_shap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs('/content/models', exist_ok=True)
artifact = {
    'model': best_model, 'scaler': scaler,
    'label_encoder': le_y, 'trimester_encoder': le_trim,
    'feature_names': FEATURE_COLS,
    'class_labels': list(le_y.classes_),
    'best_model_name': best_name,
    'test_accuracy': float(acc),
    'test_f1_macro': float(f1),
    'version': 'v4',
    'total_risks': 42,
    'dataset': {'global':3000,'uzbekistan':2000,'total':len(df_all)},
    'new_features_v4': ['thyroid_symptoms','rh_checked','painless_bleeding',
                        'fasting_glucose','cervix_short','belly_very_large',
                        'itching_palms_soles','post_term'],
    'sources': [
        'WHO ANC 2016', 'ACOG 2023', 'FIGO 2022',
        'SMFM Consult #53 (ICP)', 'WHO/IADPSG 2013 (GDM)',
        'ATA Guidelines 2017 (Thyroid)',
        'Andijan Maternal Mortality Study 2025',
        'WHO Uzbekistan Statistics', 'Demografiya.uz 2021'
    ]
}
joblib.dump(artifact, '/content/models/onamiz_v4.joblib')
for f in ['onamiz_v4.joblib']:
    shutil.copy(f'/content/models/{f}', f'/content/drive/MyDrive/onamiz_models/{f}')
for f in ['v4_analysis.png','v4_model_results.png','v4_shap.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'/content/drive/MyDrive/onamiz_models/{f}')
print(f'✅ onamiz_v4.joblib saqlandi')
print(f'   Aniqlik: {acc:.1%} | F1: {f1:.4f}')
print(f'   Features: {len(FEATURE_COLS)} ta | Xavflar: 42 ta | Dataset: {len(df_all)} ta')

In [ ]:
def predict(inp):
    msgs = {
        'low':       ('🟢','yashil',     "Ko'rsatkichlar normal. Rejali ko'rikka boring."),
        'medium':    ('🟡','sariq',      "Ba'zi belgilar bor. Bu hafta shifokorni ko'ring."),
        'high':      ('🔴','qizil',      'BUGUN shifokorga boring.'),
        'emergency': ('🚨','favqulodda', 'TEZKOR: Hoziroq tez yordam!')
    }
    row = {c: inp.get(c,0) for c in FEATURE_COLS}
    probs = best_model.predict_proba(scaler.transform(pd.DataFrame([row])))[0]
    cls = le_y.classes_[np.argmax(probs)]
    e,r,m = msgs[cls]
    return {'risk':r,'emoji':e,'msg':m,'probs':{c:round(float(p),3) for c,p in zip(le_y.classes_,probs)}}

print('='*65)
print('  42 XAVF MODELI — TEST STSENARIYLAR')
print('='*65)

tests = [
    # YANGI xavflar uchun test
    ('🔴 Jigar xolestazi (ICP) — kechasi kaft qichiydi, uxlay olmaydi',
     {'trimester_enc':2,'age':29,'gestational_week':33,'systolic_bp':116,'diastolic_bp':74,
      'heart_rate':82,'itching_palms_soles':3,'anemia_level':0,'iron_supplement':1,
      'parity':0,'prenatal_visits':2,'rural':0,'pph_history':0}),

    ('🚨 Plasenta qoplag\' — og\'riqsiz yorqin qizil qon (T2)',
     {'trimester_enc':1,'age':31,'gestational_week':22,'systolic_bp':118,'diastolic_bp':76,
      'heart_rate':85,'painless_bleeding':2,'vaginal_bleeding':0,'anemia_level':1,
      'iron_supplement':1,'parity':1,'prenatal_visits':2,'rural':0,'pph_history':0}),

    ('🟡 Gestatsion diabet xavfi — qon shakari chegaraviy',
     {'trimester_enc':1,'age':33,'gestational_week':24,'systolic_bp':122,'diastolic_bp':80,
      'heart_rate':88,'fasting_glucose':1,'belly_very_large':1,'anemia_level':1,
      'iron_supplement':0,'parity':1,'prenatal_visits':2,'rural':0,'pph_history':0}),

    ('🟡 Tireoid belgilari — yurak tezlashdi, charchoq',
     {'trimester_enc':0,'age':26,'gestational_week':9,'systolic_bp':110,'diastolic_bp':70,
      'heart_rate':105,'thyroid_symptoms':2,'rh_checked':0,'nausea_severity':2,
      'anemia_level':0,'iron_supplement':1,'parity':0,'prenatal_visits':1,'rural':0,'pph_history':0}),

    ("🔴 42 hafta o'tdi — post-term",
     {'trimester_enc':2,'age':28,'gestational_week':42,'systolic_bp':120,'diastolic_bp':78,
      'heart_rate':84,'post_term':2,'fetal_movement_t3':1,'anemia_level':0,
      'iron_supplement':1,'parity':0,'prenatal_visits':2,'rural':0,'pph_history':0}),

    ("⚠️ 17 yoshli, qishloqda, og'ir anemiya, shifokorga bormagan",
     {'trimester_enc':0,'age':17,'gestational_week':10,'systolic_bp':108,'diastolic_bp':68,
      'heart_rate':98,'anemia_level':3,'iron_supplement':0,'parity':0,'prenatal_visits':0,
      'nutrition_poor':1,'rural':1,'dizziness':2,'rh_checked':0,'pph_history':0}),

    ('✅ Sog\'lom homilador (28 yosh, T2, hammasi normal)',
     {'trimester_enc':1,'age':28,'gestational_week':18,'systolic_bp':115,'diastolic_bp':75,
      'heart_rate':80,'anemia_level':0,'iron_supplement':1,'fasting_glucose':0,
      'itching_palms_soles':0,'painless_bleeding':0,'thyroid_symptoms':0,
      'parity':0,'prenatal_visits':2,'rural':0,'pph_history':0,'rh_checked':1}),
]

for name, inp in tests:
    r = predict(inp)
    print(f'\n📋 {name}')
    print(f"   {r['emoji']} {r['risk'].upper()}: {r['msg']}")
    print(f"   📊 {r['probs']}")

## ✅ v4 Xulosa

| | v3 | v4 (hozir) |
|---|---|---|
| Xavflar soni | 31 ta | **42 ta** |
| Features | 31 ta | **39 ta** |
| Yangi xavflar | — | ICP, Plasenta previa, GDM, Tireoid, Post-term, Polihidramnios, Bachadon bo'yni, Rh |
| Klinik chegara | — | Bile acid >40µmol/L, Glucose ≥5.1mmol/L, TSH >2.5, AFI >25cm |
| Dataset | 5000 | **5000+** |
